In [0]:
bronze_path = "/Volumes/ecommerce/default/ecommerce_raw/bronze"
silver_path = "/Volumes/ecommerce/default/ecommerce_raw/silver"
quarantine_path = "/Volumes/ecommerce/default/ecommerce_raw/quarantine/"

In [0]:
#Reading order table
orders_df = (
    spark.read
    .format("delta")
    .load(f"{bronze_path}/orders")
)

display(orders_df.limit(10))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z


In [0]:
#checking orders_df
from pyspark.sql import functions as F

print("Total Records: ", orders_df.count())

orders_df.printSchema()

Total Records:  99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
#Check null values
orders_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in orders_df.columns
    ]
).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [0]:
#check duplicates
duplicates = orders_df.groupBy(
    "order_id"
).count().filter(F.col("count") > 1)

display(duplicates)

order_id,count


In [0]:
#Data Quality Check 
from pyspark.sql import functions as F

orders_checked = orders_df.withColumn(
        "validation_reason",
        F.concat_ws(
            ", ",
            F.when(
                F.col("order_purchase_timestamp").isNull(),
                F.lit("INVALID_OR_MISSING_PURCHASE_TIMESTAMP")
            ),
            F.when(
                F.col("order_approved_at").isNotNull()
                & (
                    F.col("order_approved_at")
                    < F.col("order_purchase_timestamp")
                ),
                F.lit("APPROVAL_BEFORE_PURCHASE")
            )
        )
    )

In [0]:
#Valid and quarantine records
valid_orders = orders_checked.filter(
    F.col("validation_reason") == ""
).drop("validation_reason")

quarantined_orders = orders_checked.filter(
    F.col("validation_reason") != ""
).drop("validation_reason")

valid_orders.printSchema()

print("Total:", orders_checked.count())
print("Valid:", valid_orders.count())
print("Quarantined:", quarantined_orders.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Total: 99441
Valid: 99441
Quarantined: 0


In [0]:
#Silver Orders Table
valid_orders.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{silver_path}/orders")

In [0]:
#Reading the ORDERS Silver table
silver_orders = spark.read \
    .format("delta") \
    .load(f"{silver_path}/orders")

print("Silver orders count:", silver_orders.count())

display(silver_orders.limit(10))

silver_orders.printSchema()

Silver orders count: 99441


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
#ORDER_ITEMS
order_items_df = spark.read.format("delta").load(
    f"{bronze_path}/order_items"
)

In [0]:
#Checking the order_items_df
print(f"Total Records: {order_items_df.count()}")
order_items_df.printSchema()

display(order_items_df.limit(10))

Total Records: 112650
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.9,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.9,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.0,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.4


In [0]:
#Check null values
order_items_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in order_items_df.columns
    ]
).show()

+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



In [0]:
#Check duplicates
duplicates = order_items_df.groupBy(
    "order_id", "order_item_id"
).count().filter(F.col("count") > 1)

display(duplicates)

order_id,order_item_id,count


In [0]:
#data quality checks 
order_item_checked = order_items_df.withColumn(
    "validation_reason",
        F.concat_ws(
            ", ",
            F.when(
                F.col("price") < 0,
                "NEGATIVE_PRICE"
            ),
            F.when(
                F.col("freight_value") < 0,
                "NEGATIVE_FREIGHT_VALUE"
            ),
            F.when(
                F.col("order_item_id") <= 0,
                "INVALID_ORDER_ITEM_ID"
            )
        )
    )

In [0]:
#Valid and Quarantine Records
valid_order_items = order_item_checked.filter(
    F.col("validation_reason") == ""
).drop("validation_reason")

quarantined_order_items = order_item_checked.filter(
    F.col("validation_reason") != ""
).drop("validation_reason")

print("Total:", order_item_checked.count())
print("Valid:", valid_order_items.count())
print("Quarantined:", quarantined_order_items.count())

Total: 112650
Valid: 112650
Quarantined: 0


In [0]:
#Silver order_items table
valid_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/order_items")

In [0]:
#Reading order_items silver table
silver_order_items = spark.read \
    .format("delta") \
    .load(f"{silver_path}/order_items")

print("Silver order_items count:", silver_order_items.count())

display(silver_order_items.limit(10))

silver_order_items.printSchema()

Silver order_items count: 112650


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.9,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.9,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.0,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.4


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [0]:
#ORDER_PAYMENTS
order_payments_df = spark.read.format("delta").load(
    f"{bronze_path}/order_payments"
)

In [0]:
#checking order_payments_df
print(f"Total Records: {order_payments_df.count()}")

order_payments_df.printSchema()

display(order_payments_df.limit(10))

Total Records: 103886
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [0]:
#Checking null values
order_payments_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in order_payments_df.columns
    ]
).show()

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



In [0]:
#Checking Duplicates
duplicates = order_payments_df.groupBy(
    "order_id", "payment_sequential"
).count().filter(F.col("count") > 1)

display(duplicates)

order_id,payment_sequential,count


In [0]:
#Data Quality Check
order_payments_checked = order_payments_df.withColumn(
    "validation_reason",
        F.concat_ws(
            ", ",
            F.when(
                F.col("payment_sequential") <= 0,
                "INVALID_PAYMENT_SEQUENTIAL"
            ),
            F.when(
                F.col("payment_installments") <= 0,
                "INVALID_PAYMENT_INSTALLMENTS"
            ),
            F.when(
                F.col("payment_value") < 0,
                "NEGATIVE_PAYMENT_VALUE"
            )
        )
)


In [0]:
#Valid and Quarantine Records
valid_order_payments = order_payments_checked.filter(
    F.col("validation_reason") == ""
).drop("validation_reason")

quarantined_order_payments = order_payments_checked.filter(
    F.col("validation_reason") != ""
).drop("validation_reason")

print("Total:", order_payments_checked.count())
print("Valid:", valid_order_payments.count())
print("Quarantined:", quarantined_order_payments.count())

Total: 103886
Valid: 103884
Quarantined: 2


In [0]:
#SILVER_ORDER_PAYMENTS
valid_order_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/order_payments")

In [0]:
#Reading ORDER_PAYMENTS Silver Table
silver_order_payments = spark.read \
    .format("delta") \
    .load(f"{silver_path}/order_payments")

print("Silver order_payments count:", silver_order_payments.count())

display(silver_order_payments.limit(10))

silver_order_payments.printSchema()

Silver order_payments count: 103884


order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [0]:
#Quarantine ORDER_PAYMENTS
quarantined_order_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{quarantine_path}/order_payments")

In [0]:
#Reading Quarantine ORDER_PAYMENTS
quarantine_order_payments = spark.read \
    .format("delta") \
    .load(f"{quarantine_path}/order_payments")

print("Quarantined records:", quarantine_order_payments.count())

display(quarantine_order_payments.limit(10))

quarantine_order_payments.printSchema()

Quarantined records: 2


order_id,payment_sequential,payment_type,payment_installments,payment_value
744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [0]:
#ORDER_REVIEWS
order_reviews_df = spark.read.format("delta").load(
    f"{bronze_path}/order_reviews"
)

print("Total Records:", order_reviews_df.count())
order_reviews_df.printSchema()
display(order_reviews_df.limit(20))

Total Records: 99224
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47


In [0]:
#Review title and message can be null
#Check nulls
from pyspark.sql import functions as F

order_reviews_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in order_reviews_df.columns
    ]
).show()

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        0|       0|           0|               87656|                 58247|                   0|                      0|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [0]:
#Checking for blank strings since this df has
order_reviews_df.select(
    *[
        F.sum(
            F.when(F.trim(F.col(c)) == "", 1).otherwise(0)
        ).alias(c)
        for c in order_reviews_df.columns
    ]
).show()

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        0|       0|           0|                   2|                     9|                   0|                      0|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [0]:
#check for duplicates
duplicates = order_reviews_df.groupBy(
    "review_id"
).count().filter(F.col("count") > 1)

display(duplicates)

review_id,count
28642ce6250b94cc72bc85960aec6c62,2
a0a641414ff718ca079b3967ef5c2495,2
f4d74b17cd63ee35efa82cd2567de911,2
ecbaf1fce7d2c09bfab46f89065afeaf,2
6b1de94de0f4bd84dfc4136818242faa,2
957011305e7a4b6c8a266eeeb8e0316d,2
b34f4a786bf00b7c8486141ba482783b,2
d433c252647c51309432ca0b763f969b,2
5f35bbde2b32b8617aab15b7a9a0ae24,2
c5976a5a98e854fb23d7e03c6754ae60,2


In [0]:
#Used when there was issue with data parsing and comments were not parsed correctly and read in review_id

# display(
#     order_reviews_df
#     .filter(
#         F.col("review_id").isin(
#             "Recomendo.",
#             "Parabéns",
#             "Super recomendo!!!",
#             "att"
#         )
#     )
# )

In [0]:
#Duplicate ids and rows
duplicate_ids = (     
                order_reviews_df     
                .groupBy("review_id")     
                .count()     
                .filter(F.col("count") > 1) 
                ) 
duplicate_rows = order_reviews_df.join(     
                    duplicate_ids.select("review_id"),     
                    on="review_id",     
                    how="inner" 
                )

In [0]:
#Check how many different orders each duplicated review ID has
duplicate_order_check = (
    duplicate_rows
    .groupBy("review_id")
    .agg(
        F.countDistinct("order_id").alias("distinct_orders"),
        F.count("*").alias("row_count")
    )
)

display(
    duplicate_order_check
    .groupBy("distinct_orders")
    .count()
    .orderBy("distinct_orders")
)

distinct_orders,count
2,764
3,25


In [0]:
#Display a sample of the records
multiple_order_ids = (
    duplicate_rows
    .groupBy("review_id")
    .agg(
        F.countDistinct("order_id").alias("distinct_orders")
    )
    .filter(F.col("distinct_orders") > 1)
    .select("review_id")
)

display(
    duplicate_rows
    .join(multiple_order_ids, on="review_id", how="inner")
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    )
    .orderBy("review_id", "order_id")
    .limit(20)
)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,null,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07 00:00:00,2018-03-20 18:08:23
00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,null,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07 00:00:00,2018-03-20 18:08:23
0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,null,null,2017-09-21 00:00:00,2017-09-26 03:27:47
0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,null,null,2017-09-21 00:00:00,2017-09-26 03:27:47
0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,null,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07 00:00:00,2018-03-08 03:00:53
0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,null,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07 00:00:00,2018-03-08 03:00:53
017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,null,null,2018-03-02 00:00:00,2018-03-05 01:43:30
017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,null,null,2018-03-02 00:00:00,2018-03-05 01:43:30
0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,null,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o status aparece como entregue. Solicito providências",2017-09-09 00:00:00,2017-09-13 09:52:44
0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,null,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o status aparece como entregue. Solicito providências",2017-09-09 00:00:00,2017-09-13 09:52:44


In [0]:
#check whether the repeated rows are exact copies except for order_id.
duplicate_pattern = (
    duplicate_rows
    .groupBy(
        "review_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.count("*").alias("row_count")
    )
)

display(
    duplicate_pattern
    .orderBy(F.desc("row_count"))
    .limit(20)
)

review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_count,row_count
0c76e7a547a531e7bf9f0b99cba071c1,5,null,null,2017-08-31 00:00:00,2017-09-05 15:27:17,3,3
4548534449b1f572e357211b90724f1b,1,null,quero devolver o produto comprei um veio outro;;; quero meu dinheiro de volta.,2018-08-18 00:00:00,2018-08-19 00:39:54,3,3
38821b5c496b678cf91acc34892805ad,5,null,null,2017-09-03 00:00:00,2017-09-05 12:12:51,3,3
7b606b0d57b078384f0b58eac1d41d78,5,null,null,2017-02-15 00:00:00,2017-02-21 23:30:22,3,3
3415c9f764e478409e8e0660ae816dd2,5,null,null,2018-07-31 00:00:00,2018-07-31 23:11:57,3,3
9e25d6e3025e9b9a0fc7f03588d33e2b,1,Razoavel,Peço um produto por um código e vem outro totalmente diferente.,2018-08-31 00:00:00,2018-09-03 09:33:00,3,3
4219a80ab469e3fc9901437b73da3f75,5,null,Muito rápido a entrega!!!,2017-05-18 00:00:00,2017-05-19 19:12:36,3,3
832acec9bbf4efe65c3fb6423d8b4ed7,4,null,null,2018-05-10 00:00:00,2018-05-11 10:53:21,3,3
69a1068c3128a14994e3e422e4539e04,5,null,null,2017-09-06 00:00:00,2017-09-08 00:24:52,3,3
39b4603793c1c7f5f36d809b4a218664,5,null,Otimo,2017-09-01 00:00:00,2017-09-02 12:13:03,3,3


In [0]:
#Silver order reviews table
from pyspark.sql.window import Window

# Define a window for each review ID
window_spec = (
    Window
    .partitionBy("review_id")
    .orderBy("order_id")
)

# Assign a row number to each record
ranked_reviews = order_reviews_df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
)

# Keep the first record for each review ID
valid_order_reviews = (
    ranked_reviews
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# Quarantine the remaining records
quarantine_order_reviews = (
    ranked_reviews
    .filter(F.col("row_num") > 1)
    .drop("row_num")
)

In [0]:
#Validating results
print("Original rows:", order_reviews_df.count())
print("Silver rows:", valid_order_reviews.count())
print("Quarantine rows:", quarantine_order_reviews.count())

print(
    "Duplicate review IDs in Silver:",
    valid_order_reviews
    .groupBy("review_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Original rows: 99224
Silver rows: 98410
Quarantine rows: 814
Duplicate review IDs in Silver: 0


In [0]:
#Silver Order_reviews table
valid_order_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/order_reviews")

In [0]:
#Reading Order_reviews Silver Table
silver_order_reviews = spark.read \
    .format("delta") \
    .load(f"{silver_path}/order_reviews")

print("Silver order_payments count:", silver_order_reviews.count())

display(silver_order_reviews.limit(10)) 

silver_order_reviews.printSchema()

Silver order_payments count: 98410


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0001239bc1de2e33cb583967c2ca4c67,fc046d7776171871436844218f817d7d,5,null,null,2018-03-20 00:00:00,2018-03-20 18:36:04
0001cc6860aeaf5b9017fe4131a52e62,d4665434b01caa9dc3e3e78b3eb3593e,5,null,null,2018-06-22 00:00:00,2018-06-26 13:51:29
00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,5,Entrega rápida!,A entrega foi super rápida e o pendente é lindo! Igual a foto mesmo!,2018-04-25 00:00:00,2018-04-26 14:55:36
00032b0141443497c898b3093690af51,04fb47576993a3cb0c12d4b25eab6e4e,5,null,null,2017-05-30 00:00:00,2017-06-01 23:28:55
00034d88989f9a4c393bdcaec301537f,5f358d797a49fe2f24352f73426215f6,5,null,null,2017-08-12 00:00:00,2017-08-13 19:56:53
000359bceee1b6fe876b30d35b5f9ef2,97868e4c884bc85f4501ae0a66181cc9,5,null,null,2018-02-06 00:00:00,2018-02-06 20:54:37
00041b1f51e77a663df6db3f539d694f,998b1beee068744ac833dd8b31944623,1,null,null,2018-05-24 00:00:00,2018-05-24 12:45:38
00046a69550325aea5fb89f65c7387f2,9fbda7367628952bc36c3512c46d887b,5,SUPER RECOMENDO,"GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESPERAVA.",2018-07-01 00:00:00,2018-07-02 12:32:24
0005534973388c830bb858cfba83b17b,a589caa6892ceacc6bbf2f8cc30a8ad4,5,null,otimo produto. prazo cumprido. sabor tambem muito bom!,2017-09-01 00:00:00,2017-09-02 13:04:08
00055e36e9608fe969231e551983a69c,f5fea26ab547eec920e6f8ecdc5c37e4,5,null,O produto foi entregue muito antes do esperado. Ótima agilidade.,2017-11-25 00:00:00,2017-11-27 11:37:22


root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [0]:
quarantine_order_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{quarantine_path}/order_reviews")

In [0]:
#Reading Quarantine ORDER_REVIEWS
quarantine_order_reviews = spark.read \
    .format("delta") \
    .load(f"{quarantine_path}/order_reviews")

print("Quarantined records:", quarantine_order_reviews.count())

display(quarantine_order_reviews.limit(10)) 

quarantine_order_reviews.printSchema()

Quarantined records: 814


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,null,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07 00:00:00,2018-03-20 18:08:23
0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,null,null,2017-09-21 00:00:00,2017-09-26 03:27:47
0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,null,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07 00:00:00,2018-03-08 03:00:53
017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,null,null,2018-03-02 00:00:00,2018-03-05 01:43:30
0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,null,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o status aparece como entregue. Solicito providências",2017-09-09 00:00:00,2017-09-13 09:52:44
0288d42bef3dfe36930740c9588a570f,f889a5a0b44adc29c5465b99395ac3c1,3,null,null,2017-07-29 00:00:00,2017-07-31 17:11:04
02aa7f5f75e964e3c7efa59a1f515281,db92613c074f00e53066388d48ad7512,3,null,null,2017-07-27 00:00:00,2017-07-29 15:09:50
034528cb00b6bc981847acafbdf2ae0b,e1e8e3bca903de27e9a1c72b5a5795e0,1,null,null,2017-05-14 00:00:00,2017-05-15 02:17:38
03a6a25db577d0689440933055111897,3fde8b7313af6b37b84b5c7594d7add0,5,null,Muito Bom! Gostei Bastante! Tecido Ótimo! Atendeu Minhas Espectativas! 👏👏👏 Super Recomendo! 👍,2017-12-15 00:00:00,2017-12-16 01:32:18
0467560f511c516ddaa54a60edb0c291,55b2e390d5d80ada31ad1b795ebeb087,5,null,null,2017-02-09 00:00:00,2017-02-14 15:58:44


root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [0]:
#Reading CUSTOMERS bronze table
customers_df = spark.read \
    .format("delta") \
    .load(f"{bronze_path}/customers")

In [0]:
#Checking ORDERS Table

print("Customers count:", customers_df.count())
customers_df.printSchema()
display(customers_df.limit(10))

Customers count: 99441
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [0]:
#Check Null Values
customers_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in customers_df.columns
    ]
).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [0]:
#Check Duplicates
print(
    "Duplicate customer_id:",
    customers_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate customer_unique_id:",
    customers_df.groupBy("customer_unique_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Duplicate customer_id: 0
Duplicate customer_unique_id: 2997


Customer_unique_id can be duplicate, same customer across multiple customer records.

In [0]:
#Check Empty Strings
string_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
]

for col_name in string_columns:
    print(
        col_name,
        customers_df.filter(
            F.trim(F.col(col_name)) == ""
        ).count()
    )

customer_id 0
customer_unique_id 0
customer_city 0
customer_state 0


In [0]:
#Validate customer states are consistent.
display(
    customers_df
    .groupBy("customer_state")
    .count()
    .orderBy("customer_state")
)

customer_state,count
AC,81
AL,413
AM,148
AP,68
BA,3380
CE,1336
DF,2140
ES,2033
GO,2020
MA,747


In [0]:
#CUSTOMERS Silver Table
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/customers")

In [0]:
#Reading the CUSTOMERS Silver table
silver_customers = spark.read \
    .format("delta") \
    .load(f"{silver_path}/orders")

print("Silver orders count:", silver_customers.count())

display(silver_customers.limit(10))

silver_customers.printSchema()

Silver orders count: 99441


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,validation_reason
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- validation_reason: string (nullable = true)



In [0]:
#Reading GEOLOCATION Orders
geolocation_df = (
    spark.read \
    .format("delta") \
    .load(f"{bronze_path}/geolocation")
)

In [0]:
#Checking GEOLOCATION Table
print(f"Total Count: {geolocation_df.count()}")

geolocation_df.printSchema()
display(geolocation_df.limit(10))

Total Count: 1000163
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP
1041,-23.5443921648681,-46.63949930627844,sao paulo,SP
1035,-23.541577961711493,-46.64160722329613,sao paulo,SP
1012,-23.547762303364266,-46.63536053788448,são paulo,SP
1047,-23.546273112412678,-46.64122516971552,sao paulo,SP
1013,-23.546923208436723,-46.6342636964915,sao paulo,SP
1029,-23.543769055769133,-46.63427784085132,sao paulo,SP
1011,-23.547639550320632,-46.63603162315495,sao paulo,SP


In [0]:
#Check Null Values
geolocation_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in geolocation_df.columns
    ]
).show()

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+



In [0]:
#Repeated ZIP code prefixes alone cannot be considered as duplicates because one ZIP prefix can have multiple geographic coordinates.
print(
    "Exact duplicate rows:",
    geolocation_df.count() - 
    geolocation_df.dropDuplicates().count()
)

Exact duplicate rows: 261831


In [0]:
invalid_coordinates = geolocation_df.filter(
    (F.col("geolocation_lat") < -90) |
    (F.col("geolocation_lat") > 90) |
    (F.col("geolocation_lng") < -180) |
    (F.col("geolocation_lng") > 180)
)

print("Invalid coordinates:", invalid_coordinates.count())

Invalid coordinates: 0


In [0]:
# Remove exact duplicate rows for Silver
silver_geolocation = geolocation_df.dropDuplicates()

# Identify the extra copies for quarantine
quarantine_geolocation = geolocation_df.exceptAll(
    silver_geolocation
)

In [0]:
print("Original rows:", geolocation_df.count())
print("Silver rows:", silver_geolocation.count())
print("Quarantine rows:", quarantine_geolocation.count())

print(
    "Duplicate rows in Silver:",
    silver_geolocation.count()
    - silver_geolocation.dropDuplicates().count()
)

Original rows: 1000163
Silver rows: 738332
Quarantine rows: 261831
Duplicate rows in Silver: 0


In [0]:
#Silver GEOLOCATION Table
silver_geolocation.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/geolocation")

In [0]:
#Reading GEOLOCATION Silver Table
silver_geolocation = spark.read \
    .format("delta") \
    .load(f"{silver_path}/geolocation")
print("Silver geolocation count:", silver_geolocation.count())

display(silver_geolocation.limit(10))

silver_geolocation.printSchema()

Silver geolocation count: 738332


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1011,-23.546690486964888,-46.635446958807194,são paulo,SP
1027,-23.538301166801343,-46.63076391104508,sao paulo,SP
1043,-23.544734636418685,-46.64252349042071,são paulo,SP
1050,-23.54885719288482,-46.64186497030983,sao paulo,SP
1042,-23.544667372468112,-46.64066942714524,sao paulo,SP
1003,-23.54904445268046,-46.63518261226844,sao paulo,SP
1019,-23.551889908967603,-46.63105318887411,sao paulo,SP
1004,-23.550115903139222,-46.63512161420169,sao paulo,SP
1020,-23.552494470713864,-46.631394,sao paulo,SP
1026,-23.539616057447088,-46.631187497778285,sao paulo,SP


root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [0]:
#GEOLOCATION Quarantine Table
quarantine_geolocation.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{quarantine_path}/geolocation")

In [0]:
#Reading GEOLOCATION Silver Table
quarantine_geolocation = spark.read \
    .format("delta") \
    .load(f"{quarantine_path}/geolocation")
print("Quarantine geolocation count:", quarantine_geolocation.count())

display(quarantine_geolocation.limit(10))
quarantine_geolocation.printSchema()

Quarantine geolocation count: 261831


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1003,-23.54904445268046,-46.63518261226844,sao paulo,SP
1003,-23.54904445268046,-46.63518261226844,sao paulo,SP
1003,-23.54904445268046,-46.63518261226844,sao paulo,SP
1020,-23.552494470713864,-46.631394,sao paulo,SP
1020,-23.552494470713864,-46.631394,sao paulo,SP
1020,-23.552494470713864,-46.631394,sao paulo,SP
1020,-23.552494470713864,-46.631394,sao paulo,SP
1008,-23.5452611626464,-46.63554571412088,sao paulo,SP
1017,-23.54981334999131,-46.63035027254893,são paulo,SP
1017,-23.54981334999131,-46.63035027254893,são paulo,SP


root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [0]:
#Reading SELLERS table
sellers_df = (
    spark.read \
        .format("delta") \
        .load(f"{bronze_path}/sellers")
)

In [0]:
#Checking sellers_df
print("sellers_df count:", sellers_df.count())

display(sellers_df.limit(10))

sellers_df.printSchema()

sellers_df count: 3095


seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [0]:
#Check NULL Values
sellers_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in sellers_df.columns
    ]
).show()

+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+



In [0]:
#Check Duplicates
duplicate_sellers = sellers_df.groupBy(
    "seller_id"
).count().filter(F.col("count")>1)

display(duplicate_sellers)

seller_id,count


In [0]:
#Check Empty Strings
for col in ["seller_id", "seller_city", "seller_state"]:
    print(
        col,
        sellers_df.filter(
            F.trim(F.col(col)) == ""
        ).count()
    )

seller_id 0
seller_city 0
seller_state 0


In [0]:
# Check invalid ZIP codes
print(
    "Invalid ZIP codes:",
    sellers_df.filter(
        (F.col("seller_zip_code_prefix") < 0) |
        (F.col("seller_zip_code_prefix") > 99999)
    ).count()
)

Invalid ZIP codes: 0


In [0]:
#SELLERS Silver table
sellers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/sellers")    

In [0]:
#Reading SELLERS Silver Table
silver_seller_df = spark.read \
    .format("delta") \
    .load(f"{silver_path}/sellers")

print("Silver seller count:", silver_seller_df.count())

display(silver_seller_df.limit(10))

silver_seller_df.printSchema()

Silver seller count: 3095


seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [0]:
#Reading PRODUCTS Table
products_df = (
    spark.read \
        .format("delta") \
        .load(f"{bronze_path}/products")
)

In [0]:
#Checking products_df
print("products_df count:", products_df.count())

display(products_df.limit(10))

products_df.printSchema()

products_df count: 32951


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [0]:
#Check NULL Values
products_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in products_df.columns
    ]
).show()

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



In [0]:
products_df.filter(
    F.col("product_category_name").isNull() |
    F.col("product_weight_g").isNull()
).show(20, truncate=False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|a41e356c76fab66334f36de622ecbd3a|NULL                 |NULL               |NULL                      |NULL              |650             |17               |14               |12              |
|d8dee61c2034d6d075997acef1870e9b|NULL                 |NULL               |NULL                      |NULL              |300             |16               |7                |20              |
|56139431d72cd51f19eb9f7dae4d1617|N

In [0]:
products_df.filter(
    F.col("product_weight_g").isNull()
).select(
    "product_id",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).show(truncate=False)

+--------------------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+----------------+-----------------+-----------------+----------------+
|09ff539a621711667c43eba6a3bd8466|NULL            |NULL             |NULL             |NULL            |
|5eb564652db742ff8f28759cd8d2652a|NULL            |NULL             |NULL             |NULL            |
+--------------------------------+----------------+-----------------+-----------------+----------------+



In [0]:
# Check empty strings in string columns
for col_name in ["product_id", "product_category_name"]:
    print(
        col_name,
        products_df.filter(
            F.trim(F.col(col_name)) == ""
        ).count()
    )

product_id 0
product_category_name 0


In [0]:
#Valid and quarantine records
quarantine_products = products_df.filter(
    F.col("product_id").isNotNull() &
    F.col("product_category_name").isNull() &
    F.col("product_name_lenght").isNull() &
    F.col("product_description_lenght").isNull() &
    F.col("product_photos_qty").isNull() &
    F.col("product_weight_g").isNull() &
    F.col("product_length_cm").isNull() &
    F.col("product_height_cm").isNull() &
    F.col("product_width_cm").isNull()
)

valid_products = products_df.exceptAll(quarantine_products)

print(f"Quarantine_products: {quarantine_products.count()}")
print(f"Valid_Products: {valid_products.count()}")

Quarantine_products: 1
Valid_Products: 32950


In [0]:
#PRODUCTS Silver Table
valid_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/products")

In [0]:
#Reading PRODUCTS Silver Table
silver_products_df = spark.read \
    .format("delta") \
    .load(f"{silver_path}/products")

print("Silver products count:", silver_products_df.count())

display(silver_products_df.limit(10))

silver_products_df.printSchema()

Silver products count: 32950


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
e1d1d22e9f8122a4ec1533b032c12562,ferramentas_jardim,49,1163,9,2150,70,8,34
b5cfb1d3c5e435a7a52227e08f220ee7,cool_stuff,55,410,2,1300,36,16,24
dc21247da0290904ffbbcd5953c76f95,papelaria,41,92,1,100,16,5,16
278b3c6462e86b4556b99989513ddf73,eletroportateis,58,587,3,350,20,20,20
0992c6cba95a13bfa68ea7d5e22d478b,ferramentas_jardim,42,671,1,3850,29,12,24
712a72a78ffb66cb6eee87eebbf54519,esporte_lazer,52,1049,3,400,45,22,43
f908df99196805b5bc4ada9ef510b11b,esporte_lazer,57,245,2,1950,60,15,15
5e21d5cab5d33e770d8150a4ee6117db,relogios_presentes,60,312,4,200,16,2,11
a1bf559ac1eab015ba992bd76d9d76c7,informatica_acessorios,32,1074,2,200,39,2,30
15cd31f93b25e5a517f05b1af31d353f,brinquedos,55,1475,1,400,20,20,20


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [0]:
#Quarantine PRODUCTS Table
quarantine_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{quarantine_path}/products")

In [0]:
#Reading Quarantine PRODUCTS Table
quarantine_df = spark.read \
    .format("delta") \
    .load(f"{quarantine_path}/products")

print("Quarantine products count:", quarantine_df.count())

display(quarantine_df)

quarantine_df.printSchema()

Quarantine products count: 1


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
5eb564652db742ff8f28759cd8d2652a,null,null,null,null,null,null,null,null


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [0]:
#Reading CATEGORY_TRANSLATION df
category_translation_df = spark.read \
    .format("delta") \
    .load(f"{bronze_path}/category_translation")


In [0]:
#Checking Category_df
print("Category count:", category_translation_df.count())

display(category_translation_df)

category_translation_df.printSchema()

Category count: 71


product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [0]:
#Check NULL Values
category_translation_df.select(
    *[
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in category_translation_df.columns
    ]    
).show()

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



In [0]:
#Check duplicates
category_translation_df.groupBy(
    "product_category_name", "product_category_name_english"
).count().filter(F.col("count") > 1).show()

+---------------------+-----------------------------+-----+
|product_category_name|product_category_name_english|count|
+---------------------+-----------------------------+-----+
+---------------------+-----------------------------+-----+



In [0]:
#check for empty strings
for col in category_translation_df.columns:
    print(
        col,
        category_translation_df.filter(
            F.trim(F.col(col)) == ""
        ).count()
    )

product_category_name 0
product_category_name_english 0


In [0]:
#valid and quarantine records
valid_category_translation = category_translation_df

print("Original count:", category_translation_df.count())
print("Silver count:", valid_category_translation.count())
print("Quarantine count:", 0)

Original count: 71
Silver count: 71
Quarantine count: 0


In [0]:
#CATEGORY_TRANSLATION Silver Table
valid_category_translation.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/category_translation")

In [0]:
#Reading Silver CATEGORY_TRANSLATION Table
silver_category_translation_df = spark.read \
    .format("delta") \
    .load(f"{silver_path}/category_translation")

print("Silver category_translation count:", silver_category_translation_df.count())

display(silver_category_translation_df.limit(10))

silver_category_translation_df.printSchema()

Silver category_translation count: 71


product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

